# Lab 2 · Ingestion: Auto Loader, COPY INTO and nested JSON

Land JSON files in a Unity Catalog volume, ingest them incrementally with Auto Loader, watch a new column stop and restart the stream, catch bad values in _rescued_data, prove COPY INTO is safe to rerun, and flatten nested arrays without losing rows.

**Exam:** Databricks Certified Data Engineer Associate · **Domain:** Data Ingestion and Loading (21% of the exam) · **Time:** about 35 minutes

**How to run:** attach **Serverless** compute (top right), then run the cells from top to bottom. Each part asks you to predict the result first; the expected results are listed at the end. Everything is created in the `workspace.dea_lab` schema, away from your other work.

From the Learning Hub certification guide: [Data Engineer Associate](https://www.datapreplab.com/hub#certifications/data-engineer-associate) · Independent study resource, not affiliated with or endorsed by Databricks.

## Part 1 · A schema and a landing volume

A **volume** is governed file storage in Unity Catalog. It is where raw files land before ingestion.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dea_lab;
USE workspace.dea_lab;
CREATE VOLUME IF NOT EXISTS landing COMMENT 'Raw files for the ingestion lab';

This cell resets the lab, so you can run it from the top as often as you like.

In [ ]:
base = "/Volumes/workspace/dea_lab/landing"
for folder in ("orders", "carts", "_schemas", "_checkpoints"):
    dbutils.fs.rm(f"{base}/{folder}", True)
for table in ("bronze_orders", "bronze_orders_copy"):
    spark.sql(f"DROP TABLE IF EXISTS {table}")

def land(name, lines):
    """Write one JSON Lines file into the landing folder, the way an upstream system would."""
    dbutils.fs.put(f"{base}/orders/{name}", "\n".join(lines) + "\n", True)

land("orders-001.json", [
    '{"order_id": 1, "customer": "c-10", "amount": 20.5, "country": "UK"}',
    '{"order_id": 2, "customer": "c-11", "amount": 35.0, "country": "IE"}'])
land("orders-002.json", [
    '{"order_id": 3, "customer": "c-10", "amount": "twelve", "country": "UK"}',
    '{"order_id": 4, "customer": "c-12", "amount": 8.25, "country": "FR"}'])
display(dbutils.fs.ls(f"{base}/orders"))

## Part 2 · Auto Loader, run as an incremental batch

`availableNow` processes every file not yet seen, then stops: the usual way to run a stream on a schedule. The schema location stores the inferred schema; the checkpoint remembers which files were loaded.

In [ ]:
def run_auto_loader():
    query = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{base}/_schemas/orders")
        .option("cloudFiles.schemaHints", "amount DOUBLE")       # we know amount is a number
        .load(f"{base}/orders/")
        .writeStream
        .option("checkpointLocation", f"{base}/_checkpoints/orders")
        .trigger(availableNow=True)
        .toTable("bronze_orders"))
    query.awaitTermination()
    return spark.table("bronze_orders").count()

print("Rows in bronze:", run_auto_loader())

**Predict:** what happens to order 3, whose amount is the text `"twelve"`?

In [ ]:
%sql
SELECT order_id, amount, _rescued_data FROM bronze_orders ORDER BY order_id;

In [ ]:
%sql
DESCRIBE bronze_orders;

Without a hint, Auto Loader infers JSON columns as STRING. The hint made `amount` a DOUBLE, so the value that does not fit is kept in `_rescued_data` instead of failing the load or silently disappearing.

**Predict:** run Auto Loader again with no new files. How many rows now?

In [ ]:
print("Rows in bronze:", run_auto_loader())

## Part 3 · A new column arrives

The upstream system starts sending a `coupon` field.

In [ ]:
land("orders-003.json", ['{"order_id": 5, "customer": "c-13", "amount": 15.0, "country": "UK", "coupon": "WELCOME10"}'])
try:
    run_auto_loader()
except Exception as e:
    print("The stream stopped:", str(e).splitlines()[0][:200])

With the default schema evolution mode (`addNewColumns`), Auto Loader records the new column in the schema location and **stops the stream once**. A restart (in a job, a retry) picks up the new schema:

In [ ]:
print("Rows in bronze:", run_auto_loader())
display(spark.table("bronze_orders").orderBy("order_id"))

## Part 4 · COPY INTO is safe to rerun

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS bronze_orders_copy;
COPY INTO bronze_orders_copy
FROM '/Volumes/workspace/dea_lab/landing/orders/'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

**Predict:** what does a second run load?

In [ ]:
%sql
COPY INTO bronze_orders_copy
FROM '/Volumes/workspace/dea_lab/landing/orders/'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

COPY INTO remembers the files it loaded in the table itself. To reload everything on purpose you need `COPY_OPTIONS ('force' = 'true')`, which loads duplicates.

## Part 5 · Look at files before you ingest them

`read_files` queries files directly, which is handy for checking a new source.

In [ ]:
%sql
SELECT *, _metadata.file_name
FROM read_files('/Volumes/workspace/dea_lab/landing/orders/', format => 'json');

## Part 6 · Nested JSON: explode or explode_outer?

In [ ]:
dbutils.fs.put(f"{base}/carts/carts.json", "\n".join([
    '{"cart_id": "A", "customer": {"id": "c-10", "tier": "gold"}, "items": [{"sku": "tea", "qty": 2}, {"sku": "mug", "qty": 1}]}',
    '{"cart_id": "B", "customer": {"id": "c-11", "tier": "basic"}, "items": []}',
    '{"cart_id": "C", "customer": {"id": "c-12", "tier": "basic"}}']) + "\n", True)
carts = spark.read.json(f"{base}/carts/")
carts.printSchema()

**Predict:** how many rows does each version return?

In [ ]:
from pyspark.sql import functions as F

lines = carts.select("cart_id", F.col("customer.id").alias("customer_id"), F.explode("items").alias("item"))
lines_outer = carts.select("cart_id", F.col("customer.id").alias("customer_id"), F.explode_outer("items").alias("item"))
print("explode:", lines.count(), " explode_outer:", lines_outer.count())
display(lines_outer.select("cart_id", "customer_id", "item.sku", "item.qty"))

## Expected results

```text
Part 2: 4 rows. Order 3 has amount NULL and _rescued_data holding {"amount":"twelve", ...}.
        DESCRIBE shows amount DOUBLE (the hint), every other column STRING, plus _rescued_data.
        The second run still shows 4 rows: the checkpoint knows both files were loaded.
Part 3: the first run stops with an error about a new field (coupon). The restart succeeds:
        5 rows, coupon = WELCOME10 for order 5 and NULL for the older rows.
Part 4: the first COPY INTO loads 5 rows; the second loads 0 (num_affected_rows = 0).
Part 5: 5 rows, with the file each row came from.
Part 6: explode returns 2 rows (cart A only). explode_outer returns 4: cart A twice, then
        carts B and C once each with a NULL item.
```

## Exam links

- New files only, on a schedule, millions of files or a stream: Auto Loader. Thousands of files from SQL, rerun safely: COPY INTO.
- A new checkpoint location makes Auto Loader reload every file.
- `explode` drops rows whose array is empty or NULL; `explode_outer` keeps them.
- In a job, give the Auto Loader task retries so a schema-change stop restarts on its own.